# RoomFit — Colab GPU 실행

이 노트북은 깨끗한 방 사진과 가구 참고 사진, **두 장**으로 FLUX.2 klein 4B 편집을 실행합니다. 사용자가 칠한 배치 영역은 좌표와 왼쪽·중앙·오른쪽 설명으로 프롬프트에 전달합니다. 빨간 미리보기는 기본 모델 입력에 포함하지 않습니다. **외부 추론 API나 유료 API 키는 사용하지 않습니다.** 모델 가중치를 다운로드한 다음 Colab GPU에서 실행합니다.

1. Python **3.12 이상**인 Colab 런타임에서 **런타임 → 런타임 유형 변경 → GPU**를 선택해 주세요. 첫 확인 셀이 Python·PyTorch·GPU를 검사합니다.
2. 아래 설명대로 현재 프로젝트를 ZIP으로 만들고 업로드해 주세요.
3. 패키지 설치 후 **런타임을 재시작**하고, 아래 **재시작 후 프로젝트 확인** 셀부터 실행해 주세요.
4. 앱 실행 셀의 임시 공개 Gradio 링크를 여세요. 방 사진과 가구 사진을 넣고 배치할 영역을 칠한 뒤 생성합니다. **제공된 방·침대 예제 불러오기** 버튼도 사용할 수 있습니다. 처음에는 512px를 권장합니다.

이 링크는 공개 접속 링크입니다. 링크를 아는 사람이 앱을 사용할 수 있으며, 사진은 Colab으로 전송됩니다. 런타임이 종료되면 앱과 다운로드 파일이 사라질 수 있습니다. 이 노트북의 CUDA 생성은 개발 환경에서 실제 실행하지 않았습니다.


## 1. 현재 프로젝트 ZIP 만들기

Mac의 **프로젝트 최상위 폴더(app.py가 있는 곳)**에서 아래 명령을 실행해 주세요. 최신 로컬 파일을 담으므로 GitHub에 커밋하거나 공개할 필요가 없습니다. 만들어진 `roomfit-colab.zip`을 다음 셀에서 업로드합니다. `.venv`, 캐시, 이전 버전 보관 폴더, Git 기록, `.env`는 제외합니다.

```bash
python3 - <<'ZIP'
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
root = Path.cwd()
excluded = {'.venv', '.git', '.cache', '.codex', '.agents', '.aws', '.vscode', '__pycache__', 'archive', 'archives', 'legacy', 'Output', 'outputs', 'node_modules', 'dist'}
with ZipFile('roomfit-colab.zip', 'w', ZIP_DEFLATED) as archive:
    for path in root.rglob('*'):
        relative = path.relative_to(root)
        if not path.is_file() or path.is_symlink() or any(part in excluded for part in relative.parts):
            continue
        if path.name == '.DS_Store' or path.name.startswith('.env') or path.suffix in {'.zip', '.pyc', '.safetensors'}:
            continue
        archive.write(path, relative.as_posix())
print('roomfit-colab.zip 생성 완료')
ZIP
```

In [ ]:
import sys

if sys.version_info < (3, 12):
    raise RuntimeError('NumPy 2.5.3에는 Python 3.12 이상이 필요합니다. 최신 Colab 런타임으로 전환해 주세요.')

import torch
from packaging.version import Version

if Version(torch.__version__.split('+')[0]) < Version('2.5'):
    raise RuntimeError('PyTorch 2.5 이상이 필요합니다. 최신 Colab GPU 런타임을 선택해 주세요.')
if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임을 선택해 주세요: 런타임 → 런타임 유형 변경 → GPU')
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')


In [ ]:
import io
import stat
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from google.colab import files

ROOT = Path('/content/roomfit')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('roomfit-colab.zip 한 개만 선택해 주세요.')
name, data = next(iter(uploaded.items()))
if not name.lower().endswith('.zip'):
    raise ValueError('프로젝트 ZIP 파일을 선택해 주세요.')
ROOT.mkdir(parents=True, exist_ok=True)
with ZipFile(io.BytesIO(data)) as archive:
    entries = archive.infolist()
    if sum(entry.file_size for entry in entries) > 2 * 1024**3:
        raise ValueError('ZIP을 푼 파일의 합계가 2GB를 초과합니다. 모델과 캐시를 제외해 주세요.')
    for entry in entries:
        relative = PurePosixPath(entry.filename)
        target = (ROOT / str(relative)).resolve()
        mode = entry.external_attr >> 16
        if relative.is_absolute() or '..' in relative.parts or '\\' in entry.filename or not target.is_relative_to(ROOT.resolve()) or stat.S_ISLNK(mode):
            raise ValueError('안전하지 않은 ZIP 경로 또는 심볼릭 링크가 있습니다.')
    archive.extractall(ROOT)
required = ('app.py', 'requirements-cuda.txt', 'roomfit/engine.py', 'samples/demo-room.png', 'samples/demo-bed.png')
if any(not (ROOT / name).is_file() for name in required):
    raise ValueError('ZIP에 앱 코드와 samples 예제 사진이 필요합니다. 프로젝트 최상위에서 위 명령으로 다시 압축해 주세요.')
del uploaded, data
print('프로젝트 준비 완료:', ROOT)


## 2. CUDA 패키지 설치

Colab에 설치된 CUDA PyTorch 2.5 이상은 유지합니다. 나머지 모델 런타임은 프로젝트의 고정 버전으로 설치합니다. FLUX.2 klein 4B는 공개 모델이므로 Hugging Face 로그인과 토큰 입력은 필수 사항이 아닙니다. 이 셀은 설치만 하며, 설치가 끝나면 다음 안내대로 **런타임을 재시작**해 주세요.


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements-cuda.txt')],
    check=True,
)
print('설치 완료. 런타임 메뉴에서 세션을 다시 시작한 뒤, 아래 재시작 후 프로젝트 확인 셀을 실행해 주세요.')


## 3. 런타임 재시작 후 프로젝트 확인

**설치가 완료되면 런타임 메뉴에서 세션을 다시 시작(Restart session)해 주세요.** 설치 전 GPU 확인에서 NumPy 등이 메모리에 로드되므로 새 패키지를 사용하려면 재시작해야 합니다.

재시작 후에는 아래 확인 셀부터 실행합니다. 프로젝트 파일은 보통 `/content/roomfit`에 남아 있지만 Python 변수와 import는 초기화되므로 경로를 다시 설정합니다. 런타임 삭제·초기화로 파일이 사라졌다면 위 프로젝트 ZIP 업로드 셀부터 다시 실행해 주세요. 패키지 설치도 유지되지 않았다면 설치와 재시작을 다시 진행합니다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path('/content/roomfit')
if not (ROOT / 'app.py').is_file():
    raise RuntimeError('프로젝트 파일이 없습니다. 위 ZIP 업로드 셀부터 다시 실행해 주세요.')
if sys.version_info < (3, 12):
    raise RuntimeError('Python 3.12 이상인 Colab 런타임이 필요합니다.')

import torch
from packaging.version import Version

if Version(torch.__version__.split('+')[0]) < Version('2.5'):
    raise RuntimeError('Transformers 5.18.0에는 PyTorch 2.5 이상이 필요합니다. 최신 Colab GPU 런타임을 선택해 주세요.')
if not torch.cuda.is_available():
    raise RuntimeError('Colab 런타임을 GPU로 설정해 주세요.')

from diffusers import Flux2KleinPipeline
from diffusers.quantizers import PipelineQuantizationConfig
import bitsandbytes
print('프로젝트:', ROOT)
print('CUDA 런타임 import 확인 완료. 모델 가중치는 생성 버튼을 누를 때 로딩합니다.')


## 4. 앱 실행

아래 셀이 출력하는 `https://...gradio.live` 링크에서 사용합니다. 링크는 임시 공개 링크이며 런타임이 종료되면 사용할 수 없습니다. 링크를 공유하는 대상을 확인해 주세요. 앱을 열기만 해서는 모델을 불러오지 않으며 **생성** 버튼을 누르면 다운로드와 초기 로딩이 시작됩니다. GPU 메모리가 부족하면 512px로 낮추고, 필요하면 런타임을 재시작해 주세요.

In [ ]:
import os
import sys

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import gradio as gr
from app import CSS, build_app

app = build_app()
app.launch(
    server_name='0.0.0.0', share=True, css=CSS,
    theme=gr.themes.Soft(primary_hue='orange', neutral_hue='stone'),
)


모델 선택 및 실행 제한, 라이선스와 공식 출처는 프로젝트의 `docs/MODELS.md`에 정리되어 있습니다. 기본 모델 입력 순서는 **깨끗한 방, 가구 참고 이미지**입니다. 위치를 표시한 세 번째 이미지는 CLI의 `--guide marker`/`--guide map` 실험에만 사용합니다. `marker` 실제 실험에서는 빨간 선이 결과에 남았으며 `map` 품질은 검증하지 않았습니다. 정확한 배치와 가구 형태 보존은 결과를 직접 확인해야 합니다.
